In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multi-GPU in Practice

In that section we built data parallelism by hand, and a second
GPU made the training steps of LeNet slower: each GPU had too little work
to outweigh even a small exchange of gradients. A larger model gives each
GPU more work for every byte it exchanges, so it should fare better. How
much better, once the exchange is organized well? And what can several
GPUs do for a model that is too large for any one of them?

A ResNet-18 on Fashion-MNIST images enlarged to 64×64 pixels gives each
GPU far more work than LeNet. It performs 2.2 billion multiply-adds per image, about
fifty for every byte of its 45 MB of gradients, so the work that a second
GPU takes over is large compared with the gradients that the two GPUs must
exchange. Whether this advantage survives depends on how the exchange is
organized. The hand-written loop of that section waited for
the whole backward pass before it communicated, although the gradients of
the last layers were ready long before those of the first.

It also spelled out the communication itself, a `shard_map` with a `psum`
over the flattened gradients, which every new model would need again.

Data parallelism also leaves one cost untouched: every GPU keeps a full
copy of the training state. The GPT of that section with
36 blocks instead of 18 has 1.9 billion parameters. Its weights, their
gradients, and Adam's moment estimates take 29 GiB, more than the 24 GiB
of an RTX 4090. Data parallelism would place these 29 GiB on every one of
our four GPUs, so four GPUs cannot train this model any more than one can.

We first replace the hand-written loop with each framework's
production tool for data parallelism, which starts communicating while the
backward pass is still running. We measure how ResNet-18 then scales on two
and four GPUs, and compare the measurement with the cost model of
that section. We then divide the training state among the GPUs
instead of copying it, which reduces the memory per GPU but adds
communication.
that section applies both to a transformer.

In [ ]:
# XLA moves a collective away from the computation around it, so that the
# two overlap, only when its latency-hiding scheduler is on. JAX reads
# XLA_FLAGS once, when it starts, so this cell runs before the imports.
import os
os.environ['XLA_FLAGS'] = (os.environ.get('XLA_FLAGS', '') +
                           ' --xla_gpu_enable_latency_hiding_scheduler=true')

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import json
import re
import subprocess
import sys
from jax.sharding import NamedSharding, PartitionSpec as P

Two more settings of our machine live in the environment of the notebook,
set before Python starts. `XLA_PYTHON_CLIENT_ALLOCATOR=vmm` replaces JAX's
default allocator, which reserves most of every GPU when JAX starts and
carves every array out of that block, by one that reserves nothing up
front and maps each array into one contiguous range. One step below needs
about 10 GB in one piece after many smaller arrays have come and gone, and
a later cell starts a second JAX process on the same GPUs; this allocator
lets both succeed. `NCCL_LOCAL_REGISTER=0` turns off NCCL's
buffer registration, which fails harmlessly on GPUs without peer-to-peer
access, such as ours, and would print a warning at every collective.

## Production Data Parallelism

Production data parallelism computes what the loop of
that section computes. Every GPU holds a copy of the model,
computes the gradient of the loss on its share of each minibatch, and
applies the average of the gradients of all shares. What changes is how the
work is organized, and above all when the communication starts.

Our loop was already one compiled program, and its `psum` already ran
through NCCL, which uses the ring of that section. Besides the
timing, the change is that we no longer write the communication. There is
no `shard_map` and no `psum`: the placement of the inputs alone makes the
step data-parallel, and the compiler writes the collectives.

### Overlapping Communication with the Backward Pass

The backward pass computes gradients from the last layer to the first, so
the gradients of the last layers are complete long before those of the
first. Our loop in that section nevertheless waited for the
whole backward pass before it started the allreduce, and the links between
the GPUs sat idle in the meantime. The production tools group the
gradients into *buckets* of a few tens of megabytes, in the order in which
the backward pass produces them, and start the allreduce of a bucket as
soon as its last gradient exists [@Li.Zhao.Varma.ea.2020]. While one
bucket travels between the GPUs, the GPUs compute the gradients of the
next (the figure).

![Top: the allreduce waits for the whole backward pass. Bottom: the
gradients are grouped into buckets, and each bucket's allreduce starts as
soon as the backward pass has produced it. Only the last bucket runs after
the backward pass has ended.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-ddp-overlap.svg)

The size of a bucket is a compromise between the two parts of an
allreduce's time, a fixed cost $\alpha$ per call and the bytes divided by
the bandwidth $\beta$. Small buckets start early but incur $\alpha$ many
times. A single bucket for all gradients incurs $\alpha$ once, but it waits
for the whole backward pass. Whatever the size, the last bucket holds the
gradients of the first layers, which the backward pass computes last. Its
allreduce has nothing left to overlap with, so its time adds to every step.
Overlap therefore shortens the communication of a step, but it cannot
remove it.

In JAX, the compiler implements bucketing. `jax.jit` compiles the whole
training step, the forward pass, the backward pass, and the update, into
one program, so XLA knows when each gradient is ready. It merges the many
small gradient allreduces into buffers of up to 30 MiB, its counterpart of
buckets; the flag `--xla_gpu_all_reduce_combine_threshold_bytes` sets this
limit. ResNet-18's 45 MB of gradients thus travel in two allreduces. XLA
also compiles each allreduce as an asynchronous pair of
operations, `all-reduce-start` and `all-reduce-done`, so that the GPU can
run computation that does not need the sum while the sum is in transit.
It schedules the pair apart, with computation in between, only when its
latency-hiding scheduler is on; the first cell of this section turns it
on with `--xla_gpu_enable_latency_hiding_scheduler=true`. Without it, a
profile of the step shows the gradient allreduces running on the stream
of the computation, between two of its kernels.

### The Execution Model

JAX runs one process that sees all GPUs, as in that section.
`jax.jit` compiles a step into a single program, and every GPU of a mesh
runs this program on its own part of the data, so the host dispatches one
program per step however many GPUs take part. This scheme is called
*single program, multiple data* (SPMD).

In that section the program came from a `shard_map`, in which
we wrote the per-device computation and its `psum`. Now the program is
written for one device. What makes it data-parallel is the placement of
its inputs: the parameters are replicated on every GPU of the mesh
(`P()`), and the minibatch is split along its leading axis
(`P('data')`). The compiler partitions the computation to match this
placement, so every GPU computes the gradient of
its shard. Each parameter, however, is a copy on every GPU, and the
derivative of "copy to every device" is the sum over the devices
(that section). The compiler therefore sums every parameter's
gradient across the GPUs, which is the allreduce that we wrote by hand.
JAX's partitioner, Shardy, implements the SPMD partitioning of GSPMD
[@Xu.Lee.Chen.ea.2021].

### The Training Step

Our model is the ResNet-18 of that section with a stem for small
images: a 3×3 convolution without max-pooling, followed by four stages of
two residual blocks each, with He initialization as in
that section. We save it, since later chapters train it on several
GPUs.

In [ ]:
def resnet18(num_classes, in_channels=1, *, rngs):
    """ResNet-18 for small images: a 3x3 stem without max-pooling, He
    initialization."""
    he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
    def stage(num_channels, in_channels, first=False):
        return [d2l.Residual(num_channels, use_1x1conv=not first,
                             strides=1 if first else 2,
                             in_channels=in_channels, rngs=rngs),
                d2l.Residual(num_channels, in_channels=num_channels,
                             rngs=rngs)]
    return nnx.Sequential(
        # No stem bias: the batch normalization after the conv removes it.
        # Running averages decay 0.9, two-pass variance
        nnx.Conv(in_channels, 64, (3, 3), padding='same', use_bias=False,
                 kernel_init=he, rngs=rngs),
        nnx.BatchNorm(64, momentum=0.9, use_fast_variance=False, rngs=rngs),
        nnx.relu,
        *stage(64, 64, first=True), *stage(128, 64), *stage(256, 128),
        *stage(512, 256),
        lambda X: X.mean(axis=(1, 2)),  # Global average pooling
        nnx.Linear(512, num_classes, kernel_init=he, rngs=rngs))

As in that section, every GPU keeps its own copy of the
training set, here enlarged once to 64×64 pixels, which takes 983 MB in
fp32. The input pipeline then stays out of our measurements, and the only
data that cross between GPUs are gradients. We also count the gradient
bytes $N$ that every step must sum across the GPUs:

In [ ]:
data = d2l.FashionMNIST()
X_train = jax.image.resize(data.train[0][..., None] / np.float32(255),
                           (len(data.train[0]), 64, 64, 1), 'bilinear')
y_train = jnp.asarray(data.train[1], jnp.int32)
num_params = d2l.num_params(resnet18(10, rngs=nnx.Rngs(d2l.get_key())))
N = 4 * num_params                   # Gradient bytes in fp32
print(f'{num_params / 1e6:.1f}M parameters, N = {N / 1e6:.0f} MB')

The training step itself contains no parallel code. Its argument `idx`, the
indices of the minibatch's examples, arrives split across the GPUs of the
mesh, so every GPU gathers its own examples from its copy of the training
set, and the compiler takes care of the rest. `setup` builds the model and
the optimizer and replicates them, with the training set, on the first $k$
GPUs; `minibatches` draws random minibatches and splits their indices across
the mesh. The mesh axis is of type `Auto`, which lets the compiler derive
the placement of every intermediate array, here the gathered examples, from
the placement of the inputs.

In [ ]:
def setup(k, key):
    """ResNet-18 with SGD and the training set, replicated on k GPUs."""
    mesh = jax.make_mesh((k,), ('data',), devices=jax.devices()[:k],
                         axis_types=(jax.sharding.AxisType.Auto,))
    replicated = NamedSharding(mesh, P())
    model = resnet18(10, rngs=nnx.Rngs(key))
    opt = nnx.Optimizer(model, optax.sgd(0.1), wrt=nnx.Param)
    for m in (model, opt):
        nnx.update(m, jax.device_put(nnx.state(m), replicated))
    return mesh, model, opt, jax.device_put((X_train, y_train), replicated)

def minibatches(mesh, batch_size):
    """Indices of random minibatches, split across the GPUs of the mesh."""
    rng, n = np.random.default_rng(), len(y_train)
    while True:
        perm = rng.permutation(n)
        for i in range(0, n - batch_size + 1, batch_size):
            yield jax.device_put(perm[i:i + batch_size],
                                 NamedSharding(mesh, P('data')))

@nnx.jit
def train_step(model, opt, X, y, idx):
    """One SGD step on the examples idx, written for a single device."""
    def loss_fn(model):
        return optax.softmax_cross_entropy_with_integer_labels(
            model(X[idx]), y[idx]).mean()
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    opt.update(model, grads)
    return loss

### A One-Step Check

Before timing anything, we check, as in that section, that a
step on two GPUs computes what we expect. We compare it with steps on one
GPU, from the same initialization and on the same 256 examples, in full
fp32 arithmetic so that rounding differences stay small. The layer to
watch is batch normalization (that section). It normalizes
every channel with the mean and the variance over the minibatch, and when
the minibatch is split across GPUs, these statistics can cover either one
GPU's shard or the whole minibatch.

We take one step on one GPU and one step on two GPUs, from the same
initialization and on the same 256 examples, and compare the changes of the
parameters.

In [ ]:
key = d2l.get_key()                   # One initialization for both steps
idx = np.random.default_rng().permutation(len(y_train))[:256]
updates = []
with jax.default_matmul_precision('highest'):  # Full fp32 convolutions
    for k in (1, min(2, jax.device_count())):
        mesh, model, opt, (X, y) = setup(k, key)
        before = jax.tree.map(np.asarray, nnx.state(model, nnx.Param))
        train_step(model, opt, X, y,
                   jax.device_put(idx, NamedSharding(mesh, P('data'))))
        updates.append(jax.tree.map(lambda p, b: np.asarray(p) - b,
                                    nnx.state(model, nnx.Param), before))
u1, u2 = (jax.tree.leaves(u) for u in updates)
print(f'largest update {max(np.abs(u).max() for u in u1):.1e}\n'
      f'largest difference between one and two GPUs: '
      f'{max(np.abs(a - b).max() for a, b in zip(u1, u2)):.1e}')

The two steps agree to rounding error, well under one percent of the
largest update. Under `jit`, the mean and the variance of a batch
normalization layer are means over an axis that is split across the GPUs,
so the compiler computes them over the whole minibatch and inserts an
allreduce for each. The sharded step therefore
computes the single-GPU step on all 256 examples. Lowering the step and
searching the compiled program for its allreduces shows both kinds.
`allreduce_megabytes` does the search: it finds every allreduce
instruction in the text of the compiled program and adds up the sizes of
the fp32 arrays that it returns.

In [ ]:
def allreduce_megabytes(compiled):
    """Megabytes of fp32 results of each allreduce in a compiled program,
    largest first."""
    sizes = []
    for line in compiled.as_text().splitlines():
        # A synchronous all-reduce, or the -done half of an asynchronous
        # pair, whose result type lists the summed arrays only
        if re.search(r' all-reduce(-done)?\(', line):
            result = line.split(' = ')[1].split(' all-reduce')[0]
            sizes.append(sum(4 * math.prod(map(int, dims.split(',')))
                             for dims in re.findall(r'f32\[([\d,]+)\]',
                                                    result)) / 1e6)
    return sorted(sizes, reverse=True)

In [ ]:
mesh, model, opt, (X, y) = setup(jax.device_count(), d2l.get_key())
idx = next(minibatches(mesh, 256 * mesh.size))
compiled = train_step.lower(model, opt, X, y, idx).compile()
sizes = allreduce_megabytes(compiled)  # One entry per allreduce
big = [m for m in sizes if m > 1]      # Gradient buffers, above 1 MB
print(f'{len(sizes)} allreduces: {len(big)} of '
      f'{" and ".join(f"{m:.1f}" for m in big)} MB, the other '
      f'{len(sizes) - len(big)} of {sum(sizes) - sum(big):.2f} MB together')

Two large allreduces carry almost all of the 45 MB of gradients: XLA
merged the gradients of the many parameter tensors into two buffers. The
many small allreduces carry the statistics of the batch normalization
layers, each a vector of one number per channel. They are cheap in bytes,
but each of them incurs the fixed cost $\alpha$, and each lies on the path
of the forward or the backward pass, because the next layer needs its
result.

## Scaling on Four GPUs

We measure both conventions of that section: weak
scaling, which holds the minibatch per GPU fixed while the number of GPUs
$k$ grows and reports examples per second, and strong scaling, which holds
the global minibatch fixed and reports how much sooner a step finishes. The
function `step_time(k, b)` returns the seconds per step on $k$
GPUs with $b$ examples per GPU, averaged over 50 steps after a warm-up.

In [ ]:
def step_time(k, b, num_steps=50):
    """Seconds per data-parallel step on k GPUs with b examples per GPU."""
    mesh, model, opt, (X, y) = setup(k, d2l.get_key())
    batches = minibatches(mesh, k * b)
    return d2l.Benchmark(lambda: train_step(model, opt, X, y, next(batches)),
                         warmup=10, repeats=num_steps).time

Weak scaling first, with 256 examples per GPU:

In [ ]:
ks = [k for k in (1, 2, 4) if k <= d2l.num_gpus()]
weak = {k: step_time(k, 256) for k in ks}
for k in ks:
    print(f'{k} GPU(s): {k * 256 / weak[k]:5.0f} examples/s, speedup '
          f'{k * weak[1] / weak[k]:.2f}, efficiency {weak[1] / weak[k]:.0%}')

Two GPUs process about 1.7 times as many examples per second as one, and
four GPUs a little over three times as many. The *efficiency*, the speedup divided
by $k$, falls gradually as GPUs are added. Under weak
scaling every GPU does the same work at every $k$. The efficiency is
therefore the ratio of the single-GPU step time to the step time on $k$
GPUs, and what it loses is the communication that the step does not hide.

Strong scaling holds the global minibatch at 512, so each of the $k$ GPUs
processes $512/k$ examples while the allreduce keeps its size:

In [ ]:
strong = {k: step_time(k, 512 // k) for k in ks}
for k in ks:
    print(f'{k} GPU(s), {512 // k} examples each: speedup '
          f'{strong[1] / strong[k]:.2f}, efficiency '
          f'{strong[1] / strong[k] / k:.0%}')
d2l.plot(ks, [ks, [k * weak[1] / weak[k] for k in ks],
              [strong[1] / strong[k] for k in ks]], 'GPUs', 'speedup',
         legend=['linear', 'weak (256 per GPU)', 'strong (512 in all)'],
         fmts=('k:', '-o', '--s'))

At two GPUs the two conventions give about the same speedup, because 256
and 512 examples keep an RTX 4090 about equally busy. At four GPUs they
diverge. Strong scaling leaves each GPU 128 examples per step, so the
step's computation shrinks while its communication does not, and the
efficiency falls well below that of weak scaling. The same mechanism made
LeNet slower on two GPUs; here its effect is much weaker.

### Predicted and Measured Communication

The cost model the equation of that section
predicts the time of a step on $k$ GPUs from the compute time of a shard of
$B/k$ examples, the number $m$ of allreduce calls with fixed cost $\alpha$
each, and the gradient bytes $N$ at bandwidth $\beta$:

$$
t_{\textrm{step}}(k) \approx t_{\textrm{compute}}(B/k) + m\,\alpha +
\frac{2N}{\beta}.
$$

Under weak scaling the first term does not change with $k$, so the
difference $t_{\textrm{step}}(2) - t_{\textrm{step}}(1)$ is the
communication that a step exposes. For 45 MB of gradients in a few calls,
the bandwidth term dominates. The model counts the whole allreduce, so
$2N/\beta$ is the exposed time if nothing overlaps. Overlap as in
the figure lowers the exposed time: only the allreduce of
the last bucket remains exposed, provided that the earlier buckets finish
while the backward pass still runs.

As in that section, we define $\beta$ as $2N$ divided by
the time of a large allreduce on two GPUs, so $\beta$ counts the bytes
that leave and enter a GPU together. We measure it with a bare allreduce of
256 MB per GPU, the same size as there, through the library that the
training uses:

`ALLREDUCE` times the `psum` of that section on two GPUs. NCCL
reads its configuration once per process, and we will change that
configuration below, so the measurement runs in a fresh Python process of
its own.

In [ ]:
ALLREDUCE = r'''
import json, jax
from jax import numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P
from d2l import jax as d2l

mesh = jax.make_mesh((2,), ('data',), devices=jax.devices()[:2])
x = jnp.zeros((2, 2**26), device=NamedSharding(mesh, P('data')))  # 256 MB
psum = jax.jit(jax.shard_map(lambda x: jax.lax.psum(x, 'data'), mesh=mesh,
                             in_specs=P('data'), out_specs=P('data')))
t = d2l.Benchmark(lambda: psum(x), repeats=10).time
print(json.dumps({'beta': 2 * x.nbytes / mesh.size / t}))
'''
def allreduce_beta(env=None):
    """Bandwidth of a bare 256 MB allreduce on two GPUs, in bytes/s."""
    run = subprocess.run([sys.executable, '-c', ALLREDUCE], check=True,
                         capture_output=True, text=True,
                         env={**os.environ, **(env or {})})
    return json.loads(run.stdout.splitlines()[-1])['beta']

In [ ]:
if d2l.num_gpus() >= 2:
    beta = allreduce_beta()
    print(f'beta = {beta / 1e9:.1f} GB/s; k=2: predicted 2N/beta = '
          f'{1e3 * 2 * N / beta:.0f} ms, measured exposed '
          f'{1e3 * (weak[2] - weak[1]):.0f} ms per step')

The measured exposed time is close to $2N/\beta$, the time of an
allreduce that nothing overlaps. Overlap hides far less of the
communication on this machine than the bucket model predicts.

With full overlap a step would expose only the last of XLA's two gradient
allreduces. Their sizes, about 26 and 19 MB, put that bound at about two
fifths or three fifths of $2N/\beta$, depending on which comes last.

The bucket model makes three assumptions: every allreduce starts as soon
as its bucket is complete, it runs as fast as the bare benchmark, and the
backward kernels beside it run as fast as they do alone. A trace of one
step, which records when every kernel starts and ends, tests them
(exercise 4). On this machine the third fails: the
kernels of an allreduce take a share of the GPU's processors and memory
bandwidth, and the backward kernels beside them run more slowly.

The first assumption fails as well. A merged buffer is complete only when
the last of its gradients is, so in XLA's schedule of the step on two GPUs
the larger allreduce starts two convolutions before the end of the
backward pass, and the smaller one after it.

On a machine whose GPUs share an NVLink domain, each GPU exchanges data
with the others at 450 GB/s in each direction on the H100 (NVLink 4) and
at 900 GB/s on the B200 (NVLink 5), hundreds of times the rate measured
here (that section). The allreduce of ResNet-18's
gradients then takes well under a millisecond, and the cost model predicts
near-linear scaling for this model with or without overlap.

### The Collective Transport

At $k = 2$ each GPU sends and receives at $\beta/2$ in each direction,
about 2.5 GB/s. A copy between two of our GPUs, staged through host memory,
can run at up to 23 GB/s (that section), about nine
times as fast. The cause lies in how NCCL moves data between GPUs that lack
peer-to-peer access. It falls back to buffers in host memory, and by
default GPU kernels copy the data into and out of these buffers with
ordinary loads and stores across PCIe. The copy engines, the units that
perform a GPU's memory copies, move the same bytes faster. An undocumented
NCCL parameter, `NCCL_SHM_USE_CUDA_MEMCPY=1`, makes NCCL use them. It
exists in the NCCL versions that our pinned frameworks bundle, and NCCL
2.31 removed it. We measure the bare allreduce with the parameter set:

In [ ]:
if d2l.num_gpus() >= 2:
    beta_ce = allreduce_beta({'NCCL_SHM_USE_CUDA_MEMCPY': '1'})
    print(f'beta = {beta / 1e9:.1f} GB/s by default, '
          f'{beta_ce / 1e9:.1f} GB/s with the copy engines '
          f'({beta_ce / beta:.1f}x)')

The bandwidth rises nearly five-fold, and with it the predicted
communication time of ResNet-18 falls from about 18 ms to about 4 ms.

jaxlib 0.12 bundles NCCL 2.30.4. Whether the training step of this
section completes with the parameter set, and how much faster it runs, is
the subject of exercise 5. We train with NCCL's
default, because the parameter is specific to these NCCL versions and to
machines without peer-to-peer access.

One environment variable changed $\beta$ nearly five-fold here. Whether such a
setting is safe has to be checked on the training run itself, not only on
a benchmark of bare collectives.

## Sharding the Training State

Data parallelism makes training faster, but it makes no room for a larger
model: every GPU holds all of the weights, the gradients, and the
optimizer state. With Adam in fp32 these take 16 bytes per parameter
(that section), so a 24 GiB card holds at most 1.6 billion
parameters before it stores a single activation. The 36-block GPT from the
start of this section has 1.9 billion parameters. At 16 bytes each they
take 29 GiB, which no single GPU of ours holds, and data parallelism would
need these 29 GiB on every GPU.

### Partitioning the Optimizer State, Gradients, and Weights

Each GPU needs a full copy of the weights only while it computes with them,
and it never needs the full gradient or the full optimizer state. The ZeRO
family of methods [@Rajbhandari.Rasley.Ruwase.ea.2020] removes the
redundant copies in three stages. Stage 1 partitions the optimizer state:
each GPU keeps Adam's moment estimates for $1/k$ of the parameters and
updates only those parameters. Stage 2 also partitions the gradients, and
stage 3 also partitions the weights. the table lists what each
GPU then holds.

<!-- tbl-caption:Training state per GPU with Adam in fp32 on $k$ GPUs, and for the GPT with 36 blocks and 1.9 billion parameters on $k = 4$. -->

| each GPU holds | weights | gradients | Adam state | bytes per parameter | GPT, $k = 4$ |
|---|---|---|---|---|---|
| data parallelism | all | all | all | 16 | 28.6 GiB |
| ZeRO stage 1 | all | all | $1/k$ | $8 + 8/k$ | 17.9 GiB |
| ZeRO stage 2 | all | $1/k$ | $1/k$ | $4 + 12/k$ | 12.5 GiB |
| ZeRO stage 3 (FSDP) | $1/k$ | $1/k$ | $1/k$ | $16/k$ | 7.1 GiB |

The collectives follow from the identity of that section: an
allreduce is a reduce-scatter followed by an all-gather. In stages 1 and
2 the reduce-scatter leaves each GPU with the summed gradient of the
parameters it owns, which is all that its optimizer needs. Each GPU then
updates its $1/k$ of the parameters, and an all-gather assembles the
updated parameters on every GPU. The two phases move the same bytes as the
allreduce of data parallelism, now separated by the update. Stages 1 and 2
therefore reduce the memory per GPU at no cost in communication. For our
GPT, stage 1 on four GPUs already brings the training state below the
24 GiB of one card.

Stage 3 is known as *fully sharded data parallelism* (FSDP)
[@Zhao.Gu.Varma.ea.2023]. Each GPU keeps only its shard of every
block's weights, and an all-gather assembles a block's weights just before
the block computes and releases them right after
(the figure). The backward pass gathers each block's
weights once more, releases them after the block's backward pass, and
reduce-scatters the block's gradients. A step thus performs two all-gathers
and one reduce-scatter of $N$ bytes, three phases where an allreduce has
two, so FSDP communicates 1.5 times as much as data parallelism. No full
copy of a block lives longer than the block's computation, so the memory
per GPU is $16/k$ bytes per parameter plus the activations and one or two
gathered blocks.

![The life of one block's weights under FSDP on one GPU. Between uses the
block is sharded, and each GPU holds $1/k$ of it; the other blocks stay
sharded throughout. An all-gather assembles the whole block for its forward
pass, after which it is released. A second all-gather assembles it for the
backward pass, after which it is released again, and a reduce-scatter
leaves each GPU with its share of the
gradients.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-fsdp-lifecycle.svg)

the table lists the four collectives that data-parallel and
sharded training use.

<!-- tbl-caption:The collective operations behind data-parallel and sharded training. -->

| collective | what every GPU ends with | where it appears |
|---|---|---|
| allreduce | the full sum | gradients in data parallelism |
| reduce-scatter | one part of the sum | gradients in ZeRO and FSDP |
| all-gather | every part, concatenated | weights in ZeRO and FSDP |
| all-to-all | a different part from each GPU | tokens routed to the experts of a mixture of experts (that section) |

In JAX, sharding the training state is a change of placement, as data
parallelism was. Instead of replicating the weights and the optimizer state
with `P()`, we split one axis of each array across the mesh, for example
with `P('data')` on the leading axis. Splitting the weights alone, however,
lets the partitioner choose a different scheme, such as matrix products on
split weights followed by an allreduce of the activations. FSDP's all-gathers and
reduce-scatters appear only when every block's activations are also
constrained to stay split along the batch, with
`jax.lax.with_sharding_constraint(x, P('data'))`.

### Memory and Communication

Sharding reduces memory and adds communication, and the cost model says
how long the communication takes. For our GPT, the fp32 gradients take
$N$ = 7.7 GB, four bytes per parameter. On $k$ GPUs, a ring allreduce
makes every GPU send $2(k-1)N/k$ bytes and receive as many
(that section), $1.5N$ in each direction at $k = 4$ against
$N$ at $k = 2$. If every GPU keeps the bandwidth $\beta$ of about 5 GB/s
that we measured on two, one allreduce of data parallelism takes
$3N/\beta$, about 5 s, and the three phases of FSDP take $4.5N/\beta$,
about 7 s.

The computation of a step is much shorter. The forward and the backward
pass take about six floating-point operations per parameter and token
(that section). Two sequences of 1,024 tokens per GPU, which
fit next to the 7.1 GiB of stage 3 with the activation checkpointing of
that section, take about
$6 \cdot 1.9 \cdot 10^9 \cdot 2048 \approx 2.4 \times 10^{13}$
operations. At the RTX 4090's full tf32 rate of about 80 TFLOP/s that is
0.3 s, and 0.4 s with the forward pass that checkpointing repeats. On our
machine, sharding lets the model train on four GPUs, at a few percent of
the speed that its computation would allow. Over NVLink 5, at 900 GB/s in
each direction, the same three phases take about 20 ms, and FSDP is the
standard way to train models whose state does not fit on one GPU.

The arithmetic also says when to shard. A model whose training state fits
on one GPU next to its activations trains fastest with data parallelism.
On a 24 GiB card with Adam in fp32 that limit lies at about a billion
parameters, fewer for long sequences. The 18-block GPT of
that section sits at this limit. Stage 3 would shrink
its 15.1 GiB per GPU to 3.8 GiB at $k = 4$ and free 11 GiB for
activations, but on our machine the extra communication would take far
more time than the larger micro-batches save.

Past the limit, ZeRO stages 1 and 2 reduce the memory at no cost in
communication, and stage 3 removes the remaining copies with 1.5 times
the communication. Splitting the weight matrices of every layer across
GPUs instead, *tensor parallelism* (exercise 8), and
the forms of parallelism that span several machines belong to the
Language Models part.

We can now train a model on all GPUs of one machine with the tools that
practitioners use, predict from the cost model what a step will cost, and
check the prediction against a measurement. Our measurements also show how
much depends on the connection between the GPUs: when they exchange data
through host memory, communication sets the pace long before the
arithmetic does. that section applies the method of this
chapter to a transformer, where precision, compilation, and data
parallelism interact.

## Summary

Production data parallelism computes what the hand-written loop computes,
but the framework writes the collectives, and the allreduce of each bucket
of gradients starts while the backward pass is still running. A one-step
check against a single GPU shows what the data-parallel step computes,
including how batch normalization treats a split minibatch. On our four
RTX 4090s ResNet-18 gains from every added GPU, but its exposed
communication is close to the cost model's time for an allreduce that
nothing overlaps: on this machine, overlap hides little of it. ZeRO and
FSDP divide the 16 bytes per parameter of the training state among the
GPUs, stages 1 and 2 at no cost in communication and stage 3 at 1.5 times
the communication of data parallelism.

## Exercises

1. [code] **Splitting the data.** `minibatches` places the indices with
   `P('data')`, which splits them across the GPUs of the mesh.
    1. Place them with `P()` instead, which gives every GPU all indices.
       Predict which collectives the compiled `train_step` then contains
       and how `step_time(2, 256)` compares with `step_time(1, 256)`, and
       check both.
    1. Restore `P('data')` and, inside `train_step`, reshard the gathered
       images `X[idx]` to `P(None, 'data')` with `jax.sharding.reshard`,
       which splits the height of every image across the GPUs. Which values
       must neighboring GPUs exchange for a $3 \times 3$ convolution, and
       which collective does that take? Find it in the compiled
       `train_step`.

2. **Weak-scaling extrapolation.** Suppose that a step on $k$ GPUs takes
   the single-GPU step time $t_1$ plus an exposed communication time that
   is proportional to the per-GPU ring traffic of
   that section, $2(k-1)N/k$.
    1. Express the weak-scaling efficiency $E(k)$ through a single
       constant, fit the constant to your measurement at $k = 2$, and
       predict $E(4)$. Compare your prediction with the measured $E(4)$
       and with the prediction of a model in which the exposed time is the
       same for every $k \geq 2$.
    1. Predict $E(8)$ for an eight-GPU machine of the same design as ours,
       on which every transfer between GPUs is staged through host memory
       (that section). How does the total ring traffic
       through host memory grow with $k$, and in which direction should
       this move a measured $E(8)$ relative to your prediction?
    1. With gradient accumulation (that section), each
       GPU runs $a$ micro-batches of 256 examples per optimizer step and
       communicates only after the last one. Starting from your measured
       $E(4)$, find the smallest $a$ that raises the efficiency at $k = 4$
       to 95%. What happens to the global minibatch, and why does that
       matter (that section)?

3. **Cost of sharding.** ● Count communication per GPU and step in units
   of $N$, the bytes of one full set of fp32 gradients.
    1. A variant of ZeRO stage 3 keeps every gathered block from its
       forward pass until its backward pass. Which collective does this
       remove, and how much of the memory saving of stage 3 does it give
       up, in bytes per parameter of the table?
    1. Mixed-precision FSDP gathers the weights in bf16 and reduce-scatters
       the gradients in bf16, while every GPU keeps its shard of the fp32
       weights and of Adam's state. Compare its bytes per step with those
       of data parallelism in fp32, and its memory per parameter with the
       last row of the table.
    1. For the GPT with 36 blocks and 1.9 billion parameters on our four
       GPUs, and with the $\beta$ that you measured, find the number of
       tokens per GPU and step at which the computation, 6 operations
       per parameter and token at 80 TFLOP/s, takes as long as the communication of
       stage 3. How many sequences of 1,024 tokens is that, and do their
       activations fit next to the training state
       (that section)? How does gradient accumulation
       change the answer when FSDP gathers the weights anew for every
       micro-batch?

4. [code] **Bucket size.** ● Model the overlap of the figure
   as follows. The backward pass takes time $t_b$ and releases the
   gradients in $n$ buckets of equal size, bucket $i$ at time $i\,t_b/n$.
   The allreduces of the buckets run one at a time and in order, and each
   takes $\alpha + \tau/n$, where $\tau = 2N/\beta$ is the communication
   time of all gradients and $\alpha$ is a fixed cost per collective. The
   exposed communication is the time from the end of the backward pass to
   the end of the last allreduce.
    1. Show that the exposed communication is

        $$t_{\text{exposed}}(n) = \max\Bigl(\alpha + \frac{\tau}{n},\;
        n\alpha + \tau - \frac{n-1}{n}\,t_b\Bigr).$$

    1. Find the $n$ that minimizes $t_{\text{exposed}}$ when $\tau$ is well
       below $t_b$, and again when $\tau > t_b$. What does a bucket that is
       too small cost, and what does one that is too large cost?
    1. XLA merges gradient allreduces up to a byte threshold, which the
       flag `--xla_gpu_all_reduce_combine_threshold_bytes` sets. In fresh
       processes, with `XLA_FLAGS` set before JAX initializes its backend,
       sweep the threshold from 1 MB to 100 MB. For each value, count the
       gradient allreduces in the compiled `train_step` and measure the
       exposed communication `step_time(2, 256) - step_time(1, 256)`.
       Estimate $t_b$ as two thirds of the single-GPU step time and $\tau$
       as $2N/\beta$. Is the measured optimum consistent with your model,
       and which $\alpha$ does it imply?
    1. At the default threshold, the exposed communication is close to
       $\tau$ rather than to the time of the last allreduce. Record a few
       steps at $k = 1$ and $k = 2$ inside `jax.profiler.trace` and open
       the traces in Perfetto. Compare the time from the first to the last
       kernel of the backward pass in the two traces, and find when each
       gradient allreduce starts and ends. Which assumption of the model
       fails on our machine?

5. [code] **Transport configuration.** ● XLA runs its collectives through
   NCCL, so the parameter `NCCL_SHM_USE_CUDA_MEMCPY=1` applies to JAX
   programs as well. It affects only the transport through host memory that
   NCCL uses between GPUs that lack peer-to-peer access. NCCL reads its
   configuration once per process, so compare settings in fresh processes,
   as `allreduce_beta` does.
    1. Run `ALLREDUCE` once with `NCCL_DEBUG=INFO`, find the transport that
       NCCL reports for each channel, and predict whether the parameter can
       change anything on your machine.
    1. Time the `psum` for payloads from 1 MB to 256 MB under both settings
       and plot the bandwidth against the payload. Where does each
       saturate, and does the five-fold gap hold at every size?
    1. Write the training of this section into a script and run it on two
       GPUs in a fresh process with the parameter set, under a time limit.
       Does training complete, and at what speed relative to the default
       transport?
    1. Read the sizes of the allreduces in the compiled `train_step`. At
       which of them does the parameter change the bandwidth?

6. [extended] **Sharded optimizer state.** ● Implement ZeRO stage 1
    [@Rajbhandari.Rasley.Ruwase.ea.2020] declaratively and measure
    what it saves. Use `optax.adam` in `nnx.Optimizer` instead of SGD, keep
    the model replicated, and place every leaf of `nnx.state(opt)` with a
    `NamedSharding` that splits the first of its axes whose length is
    divisible by $k$, replicating the leaves that have no such axis.
    1. Run `train_step` once and show the layout of a weight afterwards.
       Stage 1 needs replicated parameters at the start of every step; if
       the compiler did not keep them replicated, restore the layout inside
       the step with `jax.sharding.reshard` and a replicated
       `NamedSharding`. Check that after a few steps the parameters match
       those of replicated Adam from the same initialization.
    1. Read the collectives of the compiled step. Which collective is new
       compared with the replicated step, and is the gradient sum an
       allreduce or a reduce-scatter? Compare the bytes per GPU of the
       compiled step with the reduce-scatter and all-gather that stage 1
       needs (that section).
    1. In fresh processes, read `'bytes_in_use'` and `'peak_bytes_in_use'`
       from `jax.devices()[0].memory_stats()` after a completed step, and
       measure the step time, with and without sharding at $k = 2$ and
       $k = 4$. Account for the differences in live memory with
       the table, and compare them with the change in the
       peak.

    *Adapted from Stanford CS336,
    [Assignment 2 (Systems)](https://github.com/stanford-cs336/assignment2-systems),
    Problems optimizer_state_sharding and
    optimizer_state_sharding_accounting.*

7. [code] **Batch-norm statistics.** ● The one-step check of
   that section shows that under `jit` the statistics
   of batch normalization are computed over the global minibatch, as on a
   single GPU.
    1. Count the small allreduces in the compiled `train_step`, and account
       for their number from the `BatchNorm` layers of `resnet18` and the
       passes in which their statistics are needed. What does each of them
       carry?
    1. Call `model.eval()` before compiling, so that every `BatchNorm`
       normalizes with its running averages, and count the allreduces
       again. Measure `step_time` with and without this change at $k = 1$,
       $k = 2$ and $k = 4$. Separate what making the statistics global
       costs per step from what computing batch statistics costs on a
       single GPU, and compare the cost of global statistics with the time
       that the added bytes take at bandwidth $\beta$. Which properties of
       these collectives, other than their size, set their cost?

8. [code] **Tensor versus data parallelism.** ● Consider the MLP block
    $\mathbf{Y} = \phi(\mathbf{X}\mathbf{W}_1)\mathbf{W}_2$ with
    $\mathbf{X} \in \mathbb{R}^{n \times d}$ holding the $n$ tokens of one
    step, $\mathbf{W}_1 \in \mathbb{R}^{d \times 4d}$,
    $\mathbf{W}_2 \in \mathbb{R}^{4d \times d}$, and an elementwise
    activation $\phi$, on $k$ devices. Tensor
    parallelism (the figure) splits $\mathbf{W}_1$ by columns and
    $\mathbf{W}_2$ by rows [@shoeybi2019megatron].
    1. Show that each device can compute, without communication, an
       $n \times d$ partial result from its columns of $\mathbf{W}_1$ and
       its rows of $\mathbf{W}_2$, and that $\mathbf{Y}$ is the sum of the
       $k$ partial results. Which collective forms that sum, and which
       collective does the backward pass need for the gradient with
       respect to $\mathbf{X}$?
    1. Data parallelism on the same $k$ devices allreduces the gradients
       of $\mathbf{W}_1$ and $\mathbf{W}_2$ once per step. Count the values
       that each scheme sends per device and step, and find the number of
       tokens $n$ below which tensor parallelism sends less.
    1. Tensor parallelism is in practice confined to the GPUs of one NVLink
       domain even where it sends less. Explain why, using when each
       scheme's communication happens relative to the computation that
       needs its result.
    1. Build the mesh with `jax.make_mesh` and
       `axis_types=(jax.sharding.AxisType.Auto,)`, so that the compiler
       chooses the layout of each product. Place $\mathbf{W}_1$ with
       `P(None, 'data')`, $\mathbf{W}_2$ with `P('data', None)`, and
       $\mathbf{X}$ replicated, and show the layout of $\mathbf{W}_1$ with
       `jax.debug.visualize_array_sharding`. Then `jit` the block and its
       gradient with respect to all three inputs, and find the collectives
       of the first sub-problem in the compiled programs.

    *Adapted from Stanford CS336,
    [Assignment 2 (Systems)](https://github.com/stanford-cs336/assignment2-systems),
    Problems data_parallel_calcs and tp_calcs.*

[Discussions](https://d2l.discourse.group/t/36792)